<a href="https://colab.research.google.com/github/Marfall/AnomalyDetection-Otus-3/blob/main/AnomalyDetection_Otus_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# ОБНАРУЖЕНИЕ АНОМАЛИЙ В БАНКОВСКИХ ТРАНЗАКЦИЯХ (creditcard.csv)
# Методы: Isolation Forest, LOF, One-Class SVM
# Визуализация: t-SNE, UMAP
# =============================================================================

# -------------------- 1. УСТАНОВКА БИБЛИОТЕК -------------------------------
!pip install -q umap-learn

# -------------------- 2. ИМПОРТ --------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

print("✅ Библиотеки загружены.")

# -------------------- 3. ЗАГРУЗКА ДАННЫХ ----------------------------------
possible_names = ['creditcard.csv']
file_path = None

for name in possible_names:
    if os.path.exists(name):
        file_path = name
        break
    if os.path.exists('/content/' + name):
        file_path = '/content/' + name
        break

if file_path is None:
    print("❌ Файл creditcard.csv не найден.")
    print("➡️ Загрузите файл вручную.")
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]
    print(f"✅ Файл '{file_path}' загружен.")

df = pd.read_csv(file_path)
print(f"✅ Данные загружены, форма: {df.shape}")
print("Первые 5 строк:")
display(df.head())

# -------------------- 4. EDA -----------------------------------------------
print("\n=== ОПИСАТЕЛЬНАЯ СТАТИСТИКА ===")
display(df.describe())

print("\n=== ИНФОРМАЦИЯ О КОЛОНКАХ ===")
df.info()

print(f"\nПропуски:\n{df.isnull().sum()}")

class_counts = df['Class'].value_counts()
print("\n=== РАСПРЕДЕЛЕНИЕ КЛАССОВ ===")
print(class_counts)
contamination = class_counts[1] / len(df)
print(f"Доля аномалий: {contamination*100:.4f}%")
print(f"contamination = {contamination:.6f}")

# Гистограммы первых 5 признаков
print("\n=== ГИСТОГРАММЫ ПРИЗНАКОВ V1-V5 ===")
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(df.columns[:5]):
    ax = axes[i//3, i%3]
    ax.hist(df[col], bins=50, alpha=0.7, color='blue', edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Значение')
    ax.set_ylabel('Частота')
plt.tight_layout()
plt.show()
plt.close('all')

# -------------------- 5. ПОДГОТОВКА ДАННЫХ --------------------------------
y = df['Class'].values
X = df.drop(columns=['Class']).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("✅ Масштабирование выполнено.")

# Выборка для визуализации (ускоряет t-SNE и UMAP)
sample_size = 10000
np.random.seed(42)
indices = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
X_sample = X_scaled[indices]
y_sample = y[indices]
print(f"Выборка для визуализации: {sample_size} точек (аномалий: {sum(y_sample)})")

# -------------------- 6. МОДЕЛИ И ОЦЕНКА ----------------------------------
models = {
    'Isolation Forest': IsolationForest(contamination=contamination, random_state=42),
    'Local Outlier Factor': LocalOutlierFactor(contamination=contamination, novelty=False),
    'One-Class SVM': OneClassSVM(nu=contamination, kernel='rbf', gamma='scale')
}

print("\n" + "="*60)
print("ОЦЕНКА МОДЕЛЕЙ")
print("="*60)

for name, model in models.items():
    print(f"\n--- {name} ---")

    if name == 'Local Outlier Factor':
        y_pred_raw = model.fit_predict(X_scaled)
    else:
        model.fit(X_scaled)
        y_pred_raw = model.predict(X_scaled)

    y_pred = np.where(y_pred_raw == -1, 1, 0)

    print(classification_report(y, y_pred, target_names=['Норма', 'Аномалия']))
    print("Confusion matrix:")
    print(confusion_matrix(y, y_pred))

    if name == 'Isolation Forest':
        scores = model.decision_function(X_scaled)
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")
    elif name == 'Local Outlier Factor':
        scores = model.negative_outlier_factor_
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")
    elif name == 'One-Class SVM':
        scores = model.decision_function(X_scaled)
        roc_auc = roc_auc_score(y, -scores)
        print(f"ROC-AUC: {roc_auc:.4f}")

# -------------------- 7. ВИЗУАЛИЗАЦИЯ (t-SNE, UMAP) -----------------------
print("\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ В 2D")
print("="*60)

print("t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
X_tsne = tsne.fit_transform(X_sample)
print("t-SNE готово.")

print("UMAP...")
umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = umap_model.fit_transform(X_sample)
print("UMAP готово.")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='coolwarm', s=5, alpha=0.7)
ax.set_title('t-SNE проекция')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(*scatter.legend_elements(), title='Class', labels=['Норма', 'Аномалия'])
ax.grid(True)

ax = axes[1]
scatter = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y_sample, cmap='coolwarm', s=5, alpha=0.7)
ax.set_title('UMAP проекция')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(*scatter.legend_elements(), title='Class', labels=['Норма', 'Аномалия'])
ax.grid(True)

plt.tight_layout()
plt.show()
plt.close('all')

# -------------------- 8. ВЫВОДЫ --------------------------------------------
print("\n" + "="*60)
print("ВЫВОДЫ")
print("="*60)
print(f"Доля аномалий: {contamination*100:.4f}%")
print("Модели показывают разную эффективность. Рекомендуется использовать ансамбль.")
print("На визуализациях аномалии часто отделены, но не всегда чётко.")
print("Для улучшения качества можно изменить порог или использовать калибровку.")
print("="*60)
print("✅ СКРИПТ ВЫПОЛНЕН.")